# pipeline

> The headless correction workflow: load a decomp run manifest, resolve the shared
> graph DB, start/resume/reopen a CorrectionSession, recompute the worklist from
> deterministic signals + persisted review state, run the D14 empty-segment prune
> (first operation), and record a chainable correction run manifest -- with a
> cheapest-form HITL approval seam.

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
import json
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue

from cjm_transcript_correction_core.models import (
    CorrectionConfig, CorrectionManifest, SpineSegment, WorklistItem, new_run_id,
)
from cjm_transcript_correction_core.signals import compute_signal_flags, detect_empty_segments
from cjm_transcript_correction_core.graph import (
    load_document_segments, load_empty_segments, count_document_segments,
    build_prune_correction, commit_nodes_edges, start_session, get_session,
    set_session_status, record_review_markers, load_review_markers,
    find_corrections_for_session, project_effective_spine,
)

logger = logging.getLogger(__name__)

In [ ]:
#| export
def load_decomp_manifest(
    path: str,  # Path to a decomp-core run manifest JSON
) -> Dict[str, Any]:  # Parsed manifest dict
    """Load + lightly validate a decomp-core run manifest (untyped JSON; CR-20 interchange)."""
    data = json.loads(Path(path).read_text())
    fmt = data.get("format", "")
    if "decomp-core" not in fmt:
        logger.warning(f"unexpected decomp manifest format: {fmt!r} (continuing)")
    return data


def resolve_graph_db_path(
    manifest: Dict[str, Any],        # Decomp manifest dict
    graph_plugin: str,               # Graph-storage capability id
    override: Optional[str] = None,  # Explicit override (wins)
) -> Optional[str]:  # Absolute DB path the spine lives in
    """Resolve the graph DB path: explicit override > the decomp manifest's recorded db_path.

    The spine lives in the decomp-core graph DB; this core writes its overlay into
    the SAME DB (a shared substrate resource owned by no single core -- pass-2
    graph-DB-ownership evidence).
    """
    if override:
        return override
    rec = (manifest.get("plugins", {}) or {}).get(graph_plugin, {}) or {}
    return rec.get("db_path")

In [ ]:
#| export
def compute_worklist(
    segments: List[SpineSegment],                    # Ordered primary spine
    review_markers: Dict[str, str],                  # segment_id -> decision (persisted)
    secondary: Optional[List[SpineSegment]] = None,  # Optional second-transcriber spine (diff)
) -> List[WorklistItem]:  # Items still needing review, flagged
    """Recompute the worklist from layer-0 + signals + review state (only decisions persist).

    Segments already decided in this session (reviewed/corrected/skipped) drop out;
    everything flagged by a deterministic signal and not yet decided surfaces.
    """
    flags = compute_signal_flags(segments, secondary=secondary)
    items: List[WorklistItem] = []
    for i, s in enumerate(segments):
        if review_markers.get(s.id):  # already decided this session
            continue
        fl = flags.get(i, [])
        if fl:
            items.append(WorklistItem(segment=s, flags=fl))
    return items

In [ ]:
#| export
def confirm_seam(
    seam: str,                 # Seam label
    summary_lines: List[str],  # What the operator is accepting
    warnings: List[str],       # Tier-1 warnings
    assume_yes: bool = False,  # Headless: accept without prompting
) -> bool:  # True = proceed, False = aborted
    """HITL approval seam in its cheapest viable form (log + optional CLI prompt).

    Per-seam HITL-assist annotation (5 fields):
      1. signal: per-document summary + Tier-1 worklist flags
      2. deterministic pre-filter: compute_signal_flags (no AI)
      3. modality-bridge candidate: spectrogram / audio review (future Tier 2)
      4. authoritative verifier: re-transcribe-and-compare (future Tier 3; Gemini)
      5. flywheel capture: decisions persist as graph nodes/edges (DURABLE, unlike
         decomp's log-only seam -- the correction overlay IS the captured decision)
    input() blocks the loop; safe between stages with no jobs in flight (pass-2).
    """
    for line in summary_lines:
        logger.info(f"[{seam}] {line}")
    for w in warnings:
        logger.warning(f"[{seam}] {w}")
    if assume_yes:
        logger.info(f"[{seam}] auto-accepted (assume_yes)")
        return True
    reply = input(f"[{seam}] proceed? [Y/n] ").strip().lower()
    accepted = reply in ("", "y", "yes")
    logger.info(f"[{seam}] {'accepted' if accepted else 'ABORTED'} by operator")
    return accepted

In [ ]:
#| export
async def prune_empty_segments(
    queue: JobQueue,        # Started job queue
    cfg: CorrectionConfig,  # Run configuration
    graph_id: str,          # Graph-storage capability id
    document_id: str,       # Document being corrected
    total_count: int,       # Total segment count (for the summary)
    session_id: str,        # Owning session id
) -> Dict[str, Any]:  # {"pruned": n, "correction_id": id|None}
    """First operation: prune empty (silence) segments as one grouping correction (D14).

    Deterministic, no-human restructure proof: loads ONLY the empty segments
    (server-side filter -- scale-shaped), builds a batch grouping Correction +
    DERIVED_FROM edges, commits via the queue, and records REVIEWED markers
    (decision=corrected). Layer-0 untouched; reversible by superseding.
    """
    empties = await load_empty_segments(queue, graph_id, document_id)
    if not empties:
        logger.info(f"[prune] {document_id}: no empty segments")
        return {"pruned": 0, "correction_id": None}
    if not confirm_seam("prune-review",
                        [f"{document_id}: prune {len(empties)}/{total_count} empty segment(s)"],
                        [], assume_yes=cfg.assume_yes):
        logger.warning(f"[prune] {document_id}: declined by operator")
        return {"pruned": 0, "correction_id": None}
    node, edges = build_prune_correction(document_id, empties, session_id, actor=cfg.actor)
    await commit_nodes_edges(queue, graph_id, [node], edges)
    await record_review_markers(queue, graph_id, session_id,
                                [(s.id, "corrected") for s in empties])
    logger.info(f"[prune] {document_id}: grouping correction {node['id']} pruned {len(empties)} segment(s)")
    return {"pruned": len(empties), "correction_id": node["id"]}

In [ ]:
#| export
def collect_plugin_info(
    manager: PluginManager,   # Manager holding the loaded capabilities
    instance_ids: List[str],  # Instance ids to record
) -> Dict[str, Dict[str, Any]]:  # instance_id -> {name, version, db_path}
    """Record capability identity + data-DB pointers for the run manifest (provenance)."""
    info: Dict[str, Dict[str, Any]] = {}
    for iid in instance_ids:
        meta = (getattr(manager, "plugins", {}) or {}).get(iid)
        if meta is None:
            continue
        manifest = getattr(meta, "manifest", {}) or {}
        info[iid] = {"name": meta.name, "version": getattr(meta, "version", None),
                     "db_path": manifest.get("db_path")}
    return info

In [ ]:
#| export
async def run_correction(
    manager: PluginManager,                         # Manager with the graph capability loaded
    queue: JobQueue,                                # Started job queue
    cfg: CorrectionConfig,                          # Run configuration
    decomp_manifest_path: str,                      # Decomp run manifest to correct
    graph_db_path: str,                             # Resolved graph DB path (shared with decomp)
    run_id: Optional[str] = None,                   # Override run id
    session_id: Optional[str] = None,               # Resume/reopen an existing session
    reopen: bool = False,                           # Reopen a completed session
    secondary_manifest_path: Optional[str] = None,  # Second-transcriber decomp manifest (diff)
) -> CorrectionManifest:  # Manifest of the correction run
    """Correct every document in a decomp run manifest (prune + worklist surfacing).

    Per document: load spine -> [optional secondary spine] -> recompute worklist
    -> prune empty segments [prune-review seam] -> project effective spine ->
    record outcome. Resumable: a prior session's review markers drop already-decided
    segments from the worklist.
    """
    run_id = run_id or new_run_id()
    decomp = load_decomp_manifest(decomp_manifest_path)
    doc_ids = [d.get("document_id") for d in (decomp.get("documents", []) or []) if d.get("document_id")]

    # Optional secondary (second-transcriber) decomp manifest -> positional doc pairing.
    secondary_docs: Dict[str, str] = {}
    if secondary_manifest_path:
        sec = load_decomp_manifest(secondary_manifest_path)
        sec_ids = [d.get("document_id") for d in (sec.get("documents", []) or [])]
        for i, did in enumerate(doc_ids):
            if i < len(sec_ids):
                secondary_docs[did] = sec_ids[i]

    # Session: start fresh, or resume/reopen an existing one.
    if session_id:
        if await get_session(queue, cfg.graph_plugin, session_id) is None:
            raise SystemExit(f"session {session_id} not found in graph")
        if reopen:
            await set_session_status(queue, cfg.graph_plugin, session_id, "reopened")
        sess_id = session_id
        logger.info(f"resumed session {sess_id}")
    else:
        sess = await start_session(queue, cfg.graph_plugin, doc_ids)
        sess_id = sess.id
        logger.info(f"started session {sess_id} over {len(doc_ids)} document(s)")

    manifest = CorrectionManifest(
        run_id=run_id, created_at=time.time(), config=cfg.to_dict(),
        decomp_manifest=str(decomp_manifest_path),
        secondary_manifest=str(secondary_manifest_path) if secondary_manifest_path else None,
        graph_db_path=graph_db_path, session_id=sess_id,
        source_format=decomp.get("format", ""), source_version=decomp.get("version", ""),
        signals_used=["empty-text", "missing-timing", "boundary-missing-terminal",
                      "boundary-terminal-then-lowercase"]
        + (["transcriber-divergence"] if secondary_manifest_path else []),
    )

    for did in doc_ids:
        n = await count_document_segments(queue, cfg.graph_plugin, did)
        segments = await load_document_segments(queue, cfg.graph_plugin, did)
        secondary = None
        if did in secondary_docs:
            secondary = await load_document_segments(queue, cfg.graph_plugin, secondary_docs[did])
        markers = await load_review_markers(queue, cfg.graph_plugin, sess_id)
        worklist = compute_worklist(segments, markers, secondary=secondary)
        divergences = sum(1 for it in worklist if "transcriber-divergence" in it.flags)
        logger.info(f"[doc {did}] {n} segment(s); worklist {len(worklist)} flagged; "
                    f"{len(detect_empty_segments(segments))} empty"
                    + (f"; {divergences} transcriber-divergence vs {secondary_docs[did]}"
                       if secondary else ""))

        prune = {"pruned": 0, "correction_id": None}
        if cfg.prune_empty:
            prune = await prune_empty_segments(queue, cfg, cfg.graph_plugin, did, n, sess_id)

        corrections = await find_corrections_for_session(queue, cfg.graph_plugin, sess_id)
        effective = project_effective_spine(segments, corrections)
        manifest.documents.append({
            "document_id": did,
            "segment_count": n,
            "worklist_flagged": len(worklist),
            "empty_segments": len(detect_empty_segments(segments)),
            "secondary_document_id": secondary_docs.get(did),
            "transcriber_divergences": divergences if secondary else 0,
            "pruned": prune["pruned"],
            "prune_correction_id": prune["correction_id"],
            "effective_segment_count": len(effective),
        })

    await set_session_status(queue, cfg.graph_plugin, sess_id, "completed")
    return manifest

In [ ]:
# pipeline pure-logic checks (no runtime)
segs = [
    SpineSegment(id="a", index=0, text="The art of war"),
    SpineSegment(id="b", index=1, text=""),
    SpineSegment(id="c", index=2, text="is of vital importance."),
    SpineSegment(id="d", index=3, text="the general"),
]
wl = compute_worklist(segs, review_markers={})
ids = {it.segment.id for it in wl}
assert "b" in ids                                  # empty flagged
wl2 = compute_worklist(segs, review_markers={"b": "corrected"})
assert "b" not in {it.segment.id for it in wl2}    # decided -> dropped
m = {"plugins": {"cjm-graph-plugin-sqlite": {"db_path": "/tmp/g.db"}}}
assert resolve_graph_db_path(m, "cjm-graph-plugin-sqlite") == "/tmp/g.db"
assert resolve_graph_db_path(m, "cjm-graph-plugin-sqlite", override="/x") == "/x"
print("pipeline checks OK")

pipeline checks OK
